# Asset Manager end-to-end use case

> Integrating Bayesian Inference, LLM-based Sentiment Shocks, and Causal Attribution for Asset Lifecycle Valuation.

    > Version: v01: 2026-06-01

Asset manager is a chatbot which maintains a dashboard of a set of assets (described in data/assets.csv) and estimates its value and impact of news and lifecycle events on its valuation over time, notifying the user when peaks and valleys are detected in its valuation; it takes a description of items of interest to manage (eg, financial, IT, industry/organizational stock assets, domestic hardware, etc) from data/assets.csv and scrapes a specified set of data sources from internet data sources specified about that specific asset category and asset, and monitors assets lifecycle events over time; from time to time, with specific frequencies for each class of assets, an event occurs (or is read from data/events.csv) and the chatbot maintains a database of asset lifecycle events/activities for that asset, and the valuation of that assets over time, including some value deprecation using a decay function, value increase due to positive news and/or specific events (repair, renew, maintenance, buy_new); its main function is a notification of periodic lifecycle events that need to occur and value depreciation based on outliers from a set of managed assets.


In [1]:
import sys
import scipy
import importlib.metadata as md

print("Python:", sys.version)
print("SciPy:", scipy.__version__)

for pkg in ["pymc", "arviz", "numpy", "pytensor"]:
    try:
        print(pkg, md.version(pkg))
    except Exception as e:
        print(pkg, e)

Python: 3.9.10 (main, Jul 27 2025, 14:51:56) 
[Clang 17.0.0 (clang-1700.0.13.5)]
SciPy: 1.13.1
pymc 5.12.0
arviz 0.17.1
numpy 1.26.4
pytensor 2.19.0


In [11]:
#%pip install --upgrade scipy
# %pip install "scipy<1.13"
%pip install dowhy

     |████████████████████████████████| 403 kB 4.7 MB/s eta 0:00:01
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
     |████████████████████████████████| 1.5 MB 57.3 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 23.7 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 12.3 MB/s eta 0:00:01
     |████████████████████████████████| 301 kB 58.6 MB/s eta 0:00:01
  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
     |████████████████████████████████| 935 kB 9.3 MB/s eta 0:00:01
  Using cached scikit_learn-1.6.1-cp39-cp39-macosx_12_0_arm64.whl (11.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.12.0
    Uninstalling scipy-1.12.0:
      Successfully uninstalled scipy-1.12.0
  Attempting uninstall: scikit-learn
    Found existing installation: sci

In [8]:
import pymc as pm

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


In [1]:
# !pip install pandas numpy pymc arviz langchain openai pydantic scikit-learn dowhy plotly
#!which python
import sys
sys.path.insert(0, '')
print(sys.executable)
!{sys.executable} -m pip install pymc arviz scipy

/Users/rserban/.pyenv/versions/3.9.10/bin/python
You should consider upgrading via the '/Users/rserban/.pyenv/versions/3.9.10/bin/python -m pip install --upgrade pip' command.


In [2]:
# %pip install pymc
# %pip install -U scipy<1.13.1
# %pip install -U arviz
%pip install numpy==1.26.4
# %pip install -U \
#     scikit-learn \
#     dowhy \
#     numpy \
#     scipy

  Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.15.1 requires websockets<16.0.0,>=15.0.1, but you have websockets 12.0 which is incompatible.
dowhy 0.14 requires numpy>2.0, but you have numpy 1.26.4 which is incompatible.
causalml 0.15.2 requires Cython<=0.29.34, but you have cython 3.2.5 which is incompatible.
You should consider upgrading via the '/Users/rserban/.pyenv/versions/3.9.10/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 1. Architectural Objective

The notebook will implement a pipeline that treats asset value as a stochastic process:
V_t =V_{t−1} * ⋅e^^(−λΔt) + ∑_i EventImpact_i + ∑_j NewsShock_j + ϵ
Where λ is a Bayesian-estimated decay rate and events and shocks are quantified via an LLM.

## 2. Proposed Technical Implementation (The Workflow)

**Module 1**: Synthetic Data Orchestration & Schema Validation

**Technique**: Pydantic-based data validation.

**Method**: Create a robust data generator that simulates assets.csv and events.csv with realistic correlations (e.g., IT assets decay faster than Real Estate).

**Libraries**: pandas, numpy, pydantic.

## Module 2: Bayesian Decay Estimation

**Technique**: Bayesian Linear Regression / PyMC Probabilistic Programming.

**Method**: Instead of a hard-coded decay constant, we treat the decay rate λ as a distribution. We use historical data to perform Markov Chain Monte Carlo (MCMC) sampling to estimate the posterior distribution of depreciation per asset class.

**Libraries**: PyMC, ArviZ.

## Module 3: GenAI News Impact Quantifier (The "GenAI" Core)

**Technique**: RAG (Retrieval-Augmented Generation) + Zero-Shot Sentiment Scoring.

**Method**: Scraper that pulls news headlines. Use a specialized LLM (e.g., gpt-4o or FinBERT) to map qualitative news to a quantitative Value Multiplier (e.g., "New security flaw found in Server X" → −0.15 multiplier).
Implement a prompt chain: News Text → Sentiment → Asset Impact Score.

**Libraries**: langchain, openai or transformers, sentence-transformers.

## Module 4: Dynamic Value Fusion & Time-Series Modeling

**Technique**: State-Space Models (SSM).

**Method**: Combine the Bayesian decay, the lifecycle events (repair/renew), and the GenAI shocks into a unified time-series trajectory.

**Libraries**: statsmodels or Darts (for SOTA time series).

## Module 5: Anomaly Detection & Causal Attribution

**Technique**: Isolation Forests + CausalML (Double Machine Learning).

**Method**:

- Detection: Use an IsolationForest to detect "Valleys" (unexpected value drops).

- Attribution: Once a valley is detected, use Causal Inference (DoWhy) to determine if the drop was caused by a specific event (Treatment) or simply the cumulative effect of decay (Control).

**Libraries**: scikit-learn, dowhy, causalml.

## Module 6: Optimization & Predictive Maintenance

**Technique**: Constrained Optimization.

**Method**: Solve for the optimal time to "Renew" or "Repair" an asset to maximize its lifetime value (LTV) while minimizing cost, using a simple reward function.

**Libraries**: scipy.optimize.

## Imports and environment setup

In [9]:
import sys
# !{sys.executable} -m pip show pymc arviz scipy
import scipy
import arviz
import pymc

print("SciPy:", scipy.__version__)
print("ArviZ:", arviz.__version__)
print("PyMC:", pymc.__version__)

SciPy: 1.13.1
ArviZ: 0.17.1
PyMC: 5.12.0


In [25]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import plotly.graph_objects as go
import plotly.express as px
from pydantic import BaseModel, Field
from typing import List, Optional, Dict
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
from dowhy import CausalModel
import warnings

# Suppress warnings for a cleaner notebook experience
warnings.filterwarnings("ignore")
np.random.seed(42)

print("✅ Environment Setup Complete")

AttributeError: partially initialized module 'pytensor' has no attribute 'compile' (most likely due to a circular import)

In [4]:
import importlib.metadata as md

for p in [
    "numpy",
    "scipy",
    "pytensor",
    "pymc",
    "arviz",
    "scikit-learn",
    "dowhy"
]:
    try:
        print(f"{p:15s}", md.version(p))
    except Exception as e:
        print(f"{p:15s}", e)

numpy           1.26.4
scipy           1.13.1
pytensor        2.19.0
pymc            5.12.0
arviz           0.17.1
scikit-learn    1.6.1
dowhy           0.14


In [5]:
import pytensor
print(pytensor.__version__)

AttributeError: partially initialized module 'pytensor' has no attribute 'compile' (most likely due to a circular import)

In [6]:
%pip uninstall -y pymc pytensor arviz

Found existing installation: pymc 5.12.0
Uninstalling pymc-5.12.0:
  Successfully uninstalled pymc-5.12.0
Found existing installation: pytensor 2.19.0
Uninstalling pytensor-2.19.0:
  Successfully uninstalled pytensor-2.19.0
Found existing installation: arviz 0.17.1
Uninstalling arviz-0.17.1:
  Successfully uninstalled arviz-0.17.1
Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install --upgrade pytensor~=2.17.0
    # "pymc>=5.12" \
    # "arviz>=0.20"\
    

  Using cached pytensor-2.17.4.tar.gz (3.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for pytensor: filename=pytensor-2.17.4-cp39-cp39-macosx_15_0_arm64.whl size=1268214 sha256=455198a3c0f080aaaf153bdd1f2d080bf1752bbcb74c05e5b89101e967732cc6
  Stored in directory: /Users/rserban/Library/Caches/pip/wheels/e2/97/a9/35a18956d9f32ef08ba7c205dc7eaa318177b3d22f634400ff
Successfully built pytensor
  Attempting uninstall: pytensor
    Found existing installation: pytensor 2.19.0
    Uninstalling pytensor-2.19.0:
      Successfully uninstalled pytensor-2.19.0
You should consider upgrading via the '/Users/rserban/.pyenv/versions/3.9.10/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pytensor
print(pytensor.__file__)
print(pytensor.__version__)

AttributeError: partially initialized module 'pytensor' has no attribute 'compile' (most likely due to a circular import)

In [19]:
import os

print("cwd:", os.getcwd())

for f in os.listdir("."):
    if "pytensor" in f.lower():
        print(f)

cwd: /Users/rserban/TST/DEV/PYTHON/WORK/GITHUB_REPOS/GITHUB_KFLOW_DEV/kflow-dev/data-services-portfolio/apps/assetmanager/notebooks


In [20]:
import importlib.util

spec = importlib.util.find_spec("pytensor")
print(spec)
print(spec.origin)

ModuleSpec(name='pytensor', loader=<_frozen_importlib_external.SourceFileLoader object at 0x1336fd970>, origin='/Users/rserban/.pyenv/versions/3.9.10/lib/python3.9/site-packages/pytensor/__init__.py', submodule_search_locations=['/Users/rserban/.pyenv/versions/3.9.10/lib/python3.9/site-packages/pytensor'])
/Users/rserban/.pyenv/versions/3.9.10/lib/python3.9/site-packages/pytensor/__init__.py


In [21]:
%pip uninstall -y pymc pytensor

Found existing installation: pytensor 2.17.4
Uninstalling pytensor-2.17.4:
  Successfully uninstalled pytensor-2.17.4
Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install --no-cache-dir pymc==5.12.0

     |████████████████████████████████| 473 kB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 1.7 MB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 3.6 MB 46.8 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for pytensor: filename=pytensor-2.19.0-cp39-cp39-macosx_15_0_arm64.whl size=1282836 sha256=25b4342a05522953762bbed4241f43ee2c1f8be2094d37fb1eae4c092f46e454
  Stored in directory: /private/var/folders/k7/2jtvq5_n0zlfdyfs9vllwpww0000gn/T/pip-ephem-wheel-cache-0oomjiw6/wheels/ed/a5/68/e504887bdfc91a69602663639bbb9bea47c965a397ee3a0896
Successfully built pytensor
You should consider upgrading via the '/Users/rserban/.pyenv/versions/3.9.10/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [23]:
import importlib.util
spec = importlib.util.find_spec("pytensor")
print(spec.origin)

/Users/rserban/.pyenv/versions/3.9.10/lib/python3.9/site-packages/pytensor/__init__.py


In [24]:
import os
print(os.getcwd())
[f for f in os.listdir(".") if "pytensor" in f.lower()]

/Users/rserban/TST/DEV/PYTHON/WORK/GITHUB_REPOS/GITHUB_KFLOW_DEV/kflow-dev/data-services-portfolio/apps/assetmanager/notebooks


[]